# Chest X-Ray Pneumonia Classification
## Notebook 01 — EDA & Preprocessing

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  
**Task:** Binary classification — `NORMAL` vs `PNEUMONIA`  
**Goal:** Understand the dataset structure, check class imbalance, visualize samples, and define a preprocessing + augmentation strategy.

---
### Notebook roadmap
1. Environment setup & Kaggle download
2. Dataset structure exploration
3. Class distribution & imbalance analysis
4. Image property analysis (size, aspect ratio)
5. Sample visualization
6. Preprocessing & augmentation strategy
7. PyTorch DataLoader pipeline (ready to import in later notebooks)


## 1. Environment Setup & Kaggle Download

In [ ]:
# Install kaggle API if not already installed
!pip install kaggle -q

import os, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
from collections import Counter
import random

import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

print(f"PyTorch version : {torch.__version__}")
print(f"Torchvision     : {torchvision.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Kaggle credentials ──────────────────────────────────────────────────────
# Upload your kaggle.json from: https://www.kaggle.com/settings → API → Create New Token
from google.colab import files

print("Upload your kaggle.json file:")
uploaded = files.upload()  # select kaggle.json

# Move to expected location
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("kaggle.json placed successfully.")

In [ ]:
# ── Download & unzip dataset ─────────────────────────────────────────────────
DATA_ROOT = Path("/content/chest_xray")

if not DATA_ROOT.exists():
    print("Downloading dataset (~1.2 GB)...")
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content --unzip
    # The dataset unzips into /content/chest_xray automatically
    print("Download complete.")
else:
    print("Dataset already present, skipping download.")

# Verify structure
for split in ["train", "val", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        p = DATA_ROOT / split / cls
        count = len(list(p.glob("*.jpeg")) + list(p.glob("*.jpg")) + list(p.glob("*.png")))
        print(f"  {split:5s} / {cls:9s} : {count:5d} images")

## 2. Dataset Structure Exploration

In [ ]:
def count_images(root: Path) -> dict:
    """Return {split: {class: count}} for all splits."""
    result = {}
    for split in ["train", "val", "test"]:
        result[split] = {}
        for cls in ["NORMAL", "PNEUMONIA"]:
            p = root / split / cls
            imgs = list(p.glob("*.jpeg")) + list(p.glob("*.jpg")) + list(p.glob("*.png"))
            result[split][cls] = imgs  # store paths too
    return result

dataset_map = count_images(DATA_ROOT)

# Summary table
print(f"{'Split':<8} {'NORMAL':>8} {'PNEUMONIA':>12} {'Total':>8} {'% Pneumonia':>13}")
print("-" * 55)
for split, classes in dataset_map.items():
    n  = len(classes["NORMAL"])
    p  = len(classes["PNEUMONIA"])
    tot = n + p
    print(f"{split:<8} {n:>8} {p:>12} {tot:>8} {p/tot*100:>12.1f}%")

## 3. Class Distribution & Imbalance Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {"NORMAL": "#4C9EBE", "PNEUMONIA": "#E07B54"}

for ax, split in zip(axes, ["train", "val", "test"]):
    counts = {cls: len(paths) for cls, paths in dataset_map[split].items()}
    bars = ax.bar(counts.keys(), counts.values(),
                  color=[colors[k] for k in counts.keys()],
                  edgecolor="white", linewidth=1.5, width=0.5)
    for bar, v in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(v), ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_title(f"{split.upper()} split", fontsize=13, fontweight="bold")
    ax.set_ylabel("Image count")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(0, max(counts.values()) * 1.2)

fig.suptitle("Class Distribution per Split", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/content/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Compute imbalance ratio for training set
train_n = len(dataset_map["train"]["NORMAL"])
train_p = len(dataset_map["train"]["PNEUMONIA"])
ratio = train_p / train_n
print(f"\nTraining imbalance ratio (Pneumonia : Normal) = {ratio:.2f} : 1")
print("→ This is significant. We'll use WeightedRandomSampler to address it.")

**Observation:** The training set is heavily imbalanced (~3:1 Pneumonia:Normal). The val split is tiny (only 16 images!). We'll handle this by:
- Using `WeightedRandomSampler` during training so each class is seen equally per epoch
- Using AUC-ROC and sensitivity/specificity as primary metrics (not raw accuracy)
- Optionally merging val into train and using a proper 80/20 split

## 4. Image Property Analysis

In [ ]:
# Sample 200 images from train to measure size distribution
all_train_paths = (dataset_map["train"]["NORMAL"] +
                   dataset_map["train"]["PNEUMONIA"])
sample_paths = random.sample(all_train_paths, min(200, len(all_train_paths)))

widths, heights, aspects = [], [], []
for p in sample_paths:
    with Image.open(p) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)
        aspects.append(w / h)

print("Image size statistics (sample of 200):")
print(f"  Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"  Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"  Aspect ratio — min: {min(aspects):.2f}, max: {max(aspects):.2f}, mean: {np.mean(aspects):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color="#4C9EBE", edgecolor="white")
axes[0].set_title("Width distribution"); axes[0].set_xlabel("pixels")
axes[1].hist(aspects, bins=30, color="#E07B54", edgecolor="white")
axes[1].set_title("Aspect ratio distribution"); axes[1].set_xlabel("W / H")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("Image Size Analysis", fontweight="bold")
plt.tight_layout()
plt.savefig("/content/image_size_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Sample Visualization

In [ ]:
def show_samples(dataset_map, split="train", n_per_class=4):
    fig, axes = plt.subplots(2, n_per_class, figsize=(3 * n_per_class, 7))
    class_colors = {"NORMAL": "#4C9EBE", "PNEUMONIA": "#E07B54"}

    for row, cls in enumerate(["NORMAL", "PNEUMONIA"]):
        paths = random.sample(dataset_map[split][cls], n_per_class)
        for col, path in enumerate(paths):
            ax = axes[row][col]
            img = Image.open(path).convert("L")  # grayscale
            ax.imshow(img, cmap="gray")
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(cls, fontsize=12, fontweight="bold",
                              color=class_colors[cls], rotation=90, labelpad=10)
                ax.yaxis.set_label_position("left")
                ax.tick_params(left=False)
            # Show image dimensions
            ax.set_title(f"{img.size[0]}×{img.size[1]}", fontsize=8, color="gray")

    fig.suptitle(f"Sample images — {split.upper()} split",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(f"/content/samples_{split}.png", dpi=150, bbox_inches="tight")
    plt.show()

show_samples(dataset_map, split="train", n_per_class=4)

## 6. Preprocessing & Augmentation Strategy

Based on the EDA above, here are the decisions and their justifications:

| Decision | Choice | Reason |
|---|---|---|
| Resize | 224×224 | Standard for ImageNet-pretrained models (ResNet) |
| Channels | Convert to RGB | ResNet expects 3 channels; we duplicate the grayscale channel |
| Normalization | ImageNet mean/std | Pretrained weights were trained with these stats |
| Train augmentation | RandomHorizontalFlip, RandomRotation(±10°), ColorJitter | Realistic X-ray variation; conservative to avoid clinical implausibility |
| **No** vertical flip | — | Inverted chest X-rays are not clinically valid |
| Class imbalance | WeightedRandomSampler | Ensures balanced mini-batches without discarding data |
| Val set | Merged with train, re-split 80/20 | Original val has only 16 images — unusable for monitoring |

In [ ]:
# ── Config (single source of truth — import this dict in later notebooks) ───
CFG = {
    "data_root"    : DATA_ROOT,
    "img_size"     : 224,
    "batch_size"   : 32,
    "num_workers"  : 2,
    "seed"         : 42,
    "val_split"    : 0.20,          # fraction of train used for validation
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std" : [0.229, 0.224, 0.225],
}

print("Config:")
for k, v in CFG.items():
    print(f"  {k:20s}: {v}")

In [ ]:
# ── Transforms ───────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.Grayscale(num_output_channels=3),   # duplicate to 3ch for ResNet
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG["imagenet_mean"], std=CFG["imagenet_std"]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG["imagenet_mean"], std=CFG["imagenet_std"]),
])

print("Train transform pipeline:")
print(train_transform)
print("\nVal / Test transform pipeline:")
print(val_test_transform)

## 7. PyTorch DataLoader Pipeline

In [ ]:
from torch.utils.data import Subset, ConcatDataset
from sklearn.model_selection import train_test_split

random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

# ── Load train+val together, re-split properly ───────────────────────────────
# We temporarily use val_test_transform; train subset will get train_transform below
full_train_dataset = datasets.ImageFolder(
    root=str(DATA_ROOT / "train"),
    transform=None   # we'll assign transforms per subset
)
orig_val_dataset = datasets.ImageFolder(
    root=str(DATA_ROOT / "val"),
    transform=None
)

# Merge original train + tiny val
# Note: class_to_idx must be consistent — ImageFolder sorts alphabetically
print("Class mapping:", full_train_dataset.class_to_idx)  # {NORMAL: 0, PNEUMONIA: 1}

all_indices = list(range(len(full_train_dataset)))
all_targets = full_train_dataset.targets  # list of 0/1

train_idx, val_idx = train_test_split(
    all_indices,
    test_size=CFG["val_split"],
    stratify=all_targets,         # preserve class ratio
    random_state=CFG["seed"]
)

print(f"After re-split → Train: {len(train_idx)}, Val: {len(val_idx)}")

In [ ]:
# ── Subset datasets with correct transforms ───────────────────────────────────
class TransformSubset(torch.utils.data.Dataset):
    """Wrap a Subset and apply a transform."""
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

train_subset = Subset(full_train_dataset, train_idx)
val_subset   = Subset(full_train_dataset, val_idx)

train_ds = TransformSubset(train_subset, train_transform)
val_ds   = TransformSubset(val_subset,   val_test_transform)
test_ds  = datasets.ImageFolder(
    root=str(DATA_ROOT / "test"),
    transform=val_test_transform
)

print(f"Train dataset : {len(train_ds)} images")
print(f"Val dataset   : {len(val_ds)} images")
print(f"Test dataset  : {len(test_ds)} images")

In [ ]:
# ── WeightedRandomSampler to handle class imbalance ──────────────────────────
train_labels = [all_targets[i] for i in train_idx]
class_counts = Counter(train_labels)
print(f"Train class counts: {dict(class_counts)}")

# Weight = inverse frequency
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ── DataLoaders ──────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    sampler=sampler,              # replaces shuffle=True
    num_workers=CFG["num_workers"],
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True
)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

In [ ]:
# ── Sanity check: inspect one batch ─────────────────────────────────────────
imgs, labels = next(iter(train_loader))
print(f"Batch shape : {imgs.shape}")
print(f"Labels      : {labels.tolist()}")
print(f"Label counts in batch — 0 (NORMAL): {(labels==0).sum().item()}, "
      f"1 (PNEUMONIA): {(labels==1).sum().item()}")
print(f"Pixel range : [{imgs.min():.3f}, {imgs.max():.3f}]  (after ImageNet normalization)")

In [ ]:
# ── Visualize augmented batch ────────────────────────────────────────────────
# De-normalize for display
mean = torch.tensor(CFG["imagenet_mean"]).view(3, 1, 1)
std  = torch.tensor(CFG["imagenet_std"]).view(3, 1, 1)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
axes = axes.flatten()
class_names = ["NORMAL", "PNEUMONIA"]
label_colors = ["#4C9EBE", "#E07B54"]

for i in range(min(32, len(imgs))):
    img_display = (imgs[i] * std + mean).permute(1, 2, 0).numpy()
    img_display = np.clip(img_display, 0, 1)
    # Since all 3 channels are identical (grayscale), just show channel 0
    axes[i].imshow(img_display[:, :, 0], cmap="gray")
    axes[i].set_title(class_names[labels[i].item()],
                      fontsize=7,
                      color=label_colors[labels[i].item()],
                      fontweight="bold")
    axes[i].axis("off")

fig.suptitle("Augmented training batch (de-normalized)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/augmented_batch.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary & What's Next

### Key findings from EDA
| Finding | Implication |
|---|---|
| ~3:1 class imbalance in training | Use `WeightedRandomSampler` + AUC-ROC metric |
| Original val set has only 16 images | Re-split: stratified 80/20 from training data |
| Variable image sizes (up to 2000+ px) | Resize to 224×224 |
| Grayscale images | Duplicate to 3 channels for ImageNet-pretrained models |

### Pipeline is ready
- `train_loader`, `val_loader`, `test_loader` defined with correct transforms
- `CFG` dict centralizes all hyperparameters
- `WeightedRandomSampler` handles class imbalance

### Next → Notebook 02: Baseline CNN
We'll build a simple 4-block CNN from scratch, train it for ~10 epochs, and establish baseline metrics (accuracy, AUC-ROC, sensitivity, specificity).